# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
ds = mlc.Dataset(croissant_url)
meta = ds.metadata
print(f"{meta.name}: {meta.description}")
print(f"Version: {meta.version}. Published: {meta.datePublished}")
print(f"Authors: {[author.get('name', 'N/A') for author in getattr(meta, 'author', [])] if hasattr(meta, 'author') else 'Not listed'}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All references use `@id` fields as found in the Croissant schema.

In [ ]:
# List record sets and their field @id's
record_set_ids = []

# The Dataset contains one or more RecordSets--get them from metadata
if hasattr(meta, 'recordSet'):
    record_sets = meta.recordSet
    for rs in record_sets:
        if isinstance(rs, dict) and '@id' in rs:
            rs_id = rs['@id']
        elif isinstance(rs, str):
            rs_id = rs
        else:
            continue
        record_set_ids.append(rs_id)
        print(f"RecordSet @id: {rs_id}")
        # Try to show fields for this record set
        record_set_obj = ds.record_set(rs_id)
        if record_set_obj is not None and hasattr(record_set_obj, 'field'):
            field_ids = []
            for f in record_set_obj.field:
                # Each field is dict with @id and possibly column
                if isinstance(f, dict) and '@id' in f:
                    field_ids.append(f['@id'])
                elif isinstance(f, str):
                    field_ids.append(f)
            print(f"Fields in RecordSet {rs_id}: {field_ids}")
        else:
            print(f"No fields found for RecordSet {rs_id}")
else:
    print("No RecordSet declared in metadata.")

# If record_set_ids is empty, try to infer from the ds.record_sets attribute
if not record_set_ids:
    for rs in ds.record_sets:
        if hasattr(rs, '@id'):
            record_set_ids.append(rs['@id'])

if not record_set_ids:
    print('No record sets discovered.')

## 2a. Preview Records in the First RecordSet
Print the first few records for a selected record set using `@id` referencing.

In [ ]:
# Attempt to preview records if record set found
if record_set_ids:
    preview_rs_id = record_set_ids[0]
    print(f"Showing records for RecordSet @id: {preview_rs_id}")
    for i, rec in enumerate(ds.records(record_set=preview_rs_id)):
        print(rec)
        if i >= 2:
            print('...')
            break
else:
    print('No record sets to preview.')

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare DataFrames for all record sets
dfs = {}
for rs_id in record_set_ids:
    records = list(ds.records(record_set=rs_id))
    if records:
        dfs[rs_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for RecordSet {rs_id}. Columns:")
        print(dfs[rs_id].columns.tolist())
    else:
        print(f"No records found for RecordSet {rs_id}.")

# Show top rows for the primary/first record set (if any records were loaded)
if record_set_ids and record_set_ids[0] in dfs:
    main_rs_id = record_set_ids[0]
    print(f"Data preview for RecordSet {main_rs_id}:")
    dfs[main_rs_id].head()
else:
    print('No DataFrame available to preview.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Most fields are referenced by their `@id`. Typical mass spectrometry tables include numeric columns such as peptide counts, molecular weights, etc.

In [ ]:
# Pick a numeric column for demo; update as appropriate for your dataset
# Display numeric columns in the DataFrame for selection
import numpy as np

# Identify main record set and a numeric column to operate on
if record_set_ids and record_set_ids[0] in dfs:
    main_rs_id = record_set_ids[0]
    df = dfs[main_rs_id]
    # Heuristically choose a numeric column
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field}")
        # Filter records with numeric_field above a threshold (median)
        threshold = df[numeric_field].median()
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records ({numeric_field} > {threshold}): {len(filtered_df)} rows")
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try grouping by one non-numeric (categorical) field if available
        group_candidates = [col for col in df.select_dtypes(include='object').columns if col != numeric_field]
        if group_candidates:
            group_field = group_candidates[0]
            print(f"Grouping results by: {group_field}")
            grouped = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(by=numeric_field, ascending=False)
            print(grouped.head())
        else:
            print('No group field found for groupby analysis.')
    else:
        print('No numeric columns detected in data.')
else:
    print('No data available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. This may include histograms of peptide counts, boxplots of molecular weights, or groupwise abundance comparisons.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_context('notebook')

# Plot histogram of the primary numeric field
if record_set_ids and record_set_ids[0] in dfs:
    df = dfs[record_set_ids[0]]
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field], kde=True, bins=30)
        plt.xlabel(numeric_field)
        plt.title(f'Distribution of {numeric_field}')
        plt.show()
    else:
        print('No numeric field found to plot.')
else:
    print('No loaded data to visualize.')

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR^2 Mass Spectrometry dataset using `mlcroissant`, referencing all schema entities by their `@id`. We reviewed the available record sets, extracted and previewed data, performed basic filtering and normalization, and visualized key numeric fields. Further analysis could include advanced statistical modeling, feature engineering, or proteomics-specific informatics tasks.